# Data Exploration: AMP Activity & Toxicity Prediction

**Project:** Antimicrobial Peptide (AMP) Activity & Toxicity Prediction with Protein Language Models

**Contributors:** Wissal Boussekine, Fadhila Koroghli
**Supervisor:** Dr. Amin Khouani (ESI Algiers)

This notebook performs exploratory data analysis (EDA) on three AMP databases:
1. **DBAASP** - Database of Antimicrobial Activity and Structure of Peptides
2. **APD3** - Antimicrobial Peptide Database (v3)
3. **DRAMP** - Data Repository of Antimicrobial Peptides

### Goals:
- Understand dataset sizes and basic statistics
- Analyze sequence length distributions
- Investigate amino acid composition
- Assess class imbalance for activity and toxicity labels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (12, 6)})
%matplotlib inline

---
## 1. Load Datasets

In [ ]:
# Load DBAASP
dbaasp = pd.read_csv('../data/raw/dbaasp.csv')
print(f'DBAASP: {len(dbaasp)} records')
dbaasp.head(3)

In [ ]:
# Load APD3
apd3 = pd.read_csv('../data/raw/apd3.csv')
print(f'APD3: {len(apd3)} records')
apd3.head(3)

In [ ]:
# Load DRAMP (tab-separated)
dramp = pd.read_csv('../data/raw/dramp_general.txt', sep='\t', low_memory=False)
print(f'DRAMP: {len(dramp)} records, {len(dramp.columns)} columns')
dramp.head(3)

---
## 2. Dataset Overview

In [ ]:
overview = pd.DataFrame({
    'Dataset': ['DBAASP', 'APD3', 'DRAMP'],
    'Records': [len(dbaasp), len(apd3), len(dramp)],
    'Columns': [len(dbaasp.columns), len(apd3.columns), len(dramp.columns)]
})
overview

In [ ]:
# DBAASP columns
print('DBAASP columns:', list(dbaasp.columns))
print()
print('APD3 columns:', list(apd3.columns))
print()
print('DRAMP columns:', list(dramp.columns))

---
## 3. Sequence Length Distribution

In [ ]:
# DBAASP sequence lengths
dbaasp['seq_length'] = dbaasp['sequence'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)

# APD3 already has sequence_length
# DRAMP sequence lengths
dramp['seq_length'] = dramp['Sequence_Length']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name, data in zip(axes, ['DBAASP', 'APD3', 'DRAMP'],
                           [dbaasp['seq_length'], apd3['sequence_length'], dramp['seq_length']]):
    data = data[data > 0]
    ax.hist(data, bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(f'{name}\n(n={len(data)}, mean={data.mean():.1f}, median={data.median():.0f})')
    ax.set_xlabel('Sequence Length')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/seq_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
length_stats = pd.DataFrame({
    'Dataset': ['DBAASP', 'APD3', 'DRAMP'],
    'Min': [dbaasp['seq_length'].min(), apd3['sequence_length'].min(), dramp['seq_length'].min()],
    'Max': [dbaasp['seq_length'].max(), apd3['sequence_length'].max(), dramp['seq_length'].max()],
    'Mean': [f'{dbaasp["seq_length"].mean():.1f}', f'{apd3["sequence_length"].mean():.1f}', f'{dramp["seq_length"].mean():.1f}'],
    'Median': [dbaasp['seq_length'].median(), apd3['sequence_length'].median(), dramp['seq_length'].median()],
    'Std': [f'{dbaasp["seq_length"].std():.1f}', f'{apd3["sequence_length"].std():.1f}', f'{dramp["seq_length"].std():.1f}']
})
length_stats

---
## 4. Amino Acid Composition

In [ ]:
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'

def get_aa_composition(sequences):
    counter = Counter()
    total = 0
    for seq in sequences:
        if isinstance(seq, str):
            for aa in seq.upper():
                if aa in AMINO_ACIDS:
                    counter[aa] += 1
                    total += 1
    comp = {aa: counter.get(aa, 0) / total * 100 for aa in AMINO_ACIDS}
    return comp

comp_dbaasp = get_aa_composition(dbaasp['sequence'])
comp_apd3 = get_aa_composition(apd3['sequence'])
comp_dramp = get_aa_composition(dramp['Sequence'])

comp_df = pd.DataFrame({
    'AA': list(AMINO_ACIDS),
    'DBAASP': [comp_dbaasp[aa] for aa in AMINO_ACIDS],
    'APD3': [comp_apd3[aa] for aa in AMINO_ACIDS],
    'DRAMP': [comp_dramp[aa] for aa in AMINO_ACIDS]
})

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(AMINO_ACIDS))
width = 0.25
ax.bar(x - width, comp_df['DBAASP'], width, label='DBAASP')
ax.bar(x, comp_df['APD3'], width, label='APD3')
ax.bar(x + width, comp_df['DRAMP'], width, label='DRAMP')
ax.set_xticks(x)
ax.set_xticklabels(list(AMINO_ACIDS))
ax.set_ylabel('Frequency (%)')
ax.set_title('Amino Acid Composition Across Databases')
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/aa_composition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
comp_df

---
## 5. Physicochemical Property Analysis

Analyze sequence properties: charge, hydrophobicity, and molecular weight.

In [ ]:
# Amino acid properties
HYDROPHOBICITY_SCALE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

CHARGE_SCALE = {
    'R': 1, 'K': 1, 'D': -1, 'E': -1, 'H': 0.1
}

def compute_hydrophobicity(seq):
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    scores = [HYDROPHOBICITY_SCALE.get(aa, 0) for aa in seq.upper()]
    return np.mean(scores)

def compute_charge(seq):
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    charges = [CHARGE_SCALE.get(aa, 0) for aa in seq.upper()]
    return sum(charges)

def compute_molecular_weight(seq):
    if not isinstance(seq, str) or len(seq) == 0:
        return np.nan
    aa_weights = {
        'A': 89.09, 'R': 174.20, 'N': 132.12, 'D': 133.10, 'C': 121.15,
        'Q': 146.15, 'E': 147.13, 'G': 75.07, 'H': 155.16, 'I': 131.18,
        'L': 131.18, 'K': 146.19, 'M': 149.21, 'F': 165.19, 'P': 115.13,
        'S': 105.09, 'T': 119.12, 'W': 204.23, 'Y': 181.19, 'V': 117.15
    }
    return sum(aa_weights.get(aa, 0) for aa in seq.upper())

# Compute properties
for name, data, seq_col in [('DBAASP', dbaasp, 'sequence'), ('APD3', apd3, 'sequence'), ('DRAMP', dramp, 'Sequence')]:
    data[f'hydrophobicity_{name.lower()}'] = data[seq_col].apply(compute_hydrophobicity)
    data[f'charge_{name.lower()}'] = data[seq_col].apply(compute_charge)
    data[f'mw_{name.lower()}'] = data[seq_col].apply(compute_molecular_weight)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

properties = ['hydrophobicity', 'charge', 'mw']
titles = ['Hydrophobicity (Kyte-Doolittle)', 'Net Charge at pH 7', 'Molecular Weight (Da)']
datasets = [('DBAASP', dbaasp), ('APD3', apd3), ('DRAMP', dramp)]

for row, (prop, title) in enumerate(zip(properties, titles)):
    for col, (name, data) in enumerate(datasets):
        ax = axes[row, col]
        values = data[f'{prop}_{name.lower()}'].dropna()
        values = values[np.isfinite(values)]
        ax.hist(values, bins=40, alpha=0.7, edgecolor='black')
        ax.set_title(f'{name} - {title}')
        ax.set_xlabel(title)
        ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/physicochemical_properties.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Activity Analysis (DBAASP)

DBAASP contains peptides with known antimicrobial activity. Let's analyze the distribution.

In [ ]:
# DBAASP synthesis_type as a proxy for activity labeling
synth_counts = dbaasp['synthesis_type'].value_counts()
print('Synthesis Type Distribution (DBAASP):')
print(synth_counts)

fig, ax = plt.subplots(figsize=(10, 5))
synth_counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('DBAASP - Synthesis Type Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Synthesis Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/raw/dbaasp_synthesis_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# DBAASP complexity types
complexity_counts = dbaasp['complexity'].value_counts()
print('Complexity Distribution (DBAASP):')
print(complexity_counts)

fig, ax = plt.subplots(figsize=(8, 5))
complexity_counts.plot(kind='bar', ax=ax, edgecolor='black', color='salmon')
ax.set_title('DBAASP - Peptide Complexity Distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# DRAMP Activity analysis
dramp_activity_counts = dramp['Activity'].value_counts().head(15)
print('Top 15 Activity Types (DRAMP):')
print(dramp_activity_counts)

fig, ax = plt.subplots(figsize=(12, 5))
dramp_activity_counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('DRAMP - Top 15 Activity Types')
ax.set_ylabel('Count')
ax.set_xlabel('Activity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 7. Toxicity / Hemolysis Analysis

In [ ]:
# DRAMP Hemolytic activity
print('Hemolytic Activity Distribution (DRAMP):')
hemo_counts = dramp['Hemolytic_activity'].value_counts()
print(hemo_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
labels = ['No hemolysis info', 'Has hemolysis data']
values = [hemo_counts.get('No hemolysis information or data found in the reference(s) presented in this entry', 0),
          len(dramp) - hemo_counts.get('No hemolysis information or data found in the reference(s) presented in this entry', 0)]
axes[0].pie(values, labels=labels, autopct='%1.1f%%', startangle=90, colors=['#ff9999', '#66b3ff'])
axes[0].set_title('DRAMP - Hemolysis Data Availability')

# Among those with data, what are the values?
has_hemo = dramp[dramp['Hemolytic_activity'] != 'No hemolysis information or data found in the reference(s) presented in this entry']
hemo_vals = has_hemo['Hemolytic_activity'].value_counts().head(10)
axes[1].barh(range(len(hemo_vals)), hemo_vals.values)
axes[1].set_yticks(range(len(hemo_vals)))
axes[1].set_yticklabels(hemo_vals.index.str[:50])
axes[1].set_title('Top Hemolysis Values (DRAMP)')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/hemolysis_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# DRAMP Cytotoxicity analysis
print('Cytotoxicity Distribution (DRAMP):')
cyto = dramp['Cytotoxicity'].value_counts()
print(cyto.head(15))

has_cyto = len(dramp) - cyto.get('No cytotoxicity information found', 0)
no_cyto = cyto.get('No cytotoxicity information found', 0)
print(f'\nWith cytotoxicity data: {has_cyto} ({has_cyto/len(dramp)*100:.1f}%)')
print(f'Without cytotoxicity data: {no_cyto} ({no_cyto/len(dramp)*100:.1f}%)')

---
## 8. Class Imbalance Analysis

In [ ]:
def imbalance_ratio(counts):
    if len(counts) < 2:
        return float('inf')
    sorted_counts = sorted(counts.values, reverse=True)
    return sorted_counts[0] / sorted_counts[1]

imbalance_data = []

# DBAASP: synthesis_type as proxy
synth_counts = dbaasp['synthesis_type'].value_counts()
imbalance_data.append({'Dataset': 'DBAASP', 'Label': 'synthesis_type',
                       'Majority': synth_counts.index[0], 'Majority_Count': synth_counts.iloc[0],
                       'Minority': synth_counts.index[-1] if len(synth_counts) > 1 else 'N/A',
                       'Minority_Count': synth_counts.iloc[-1] if len(synth_counts) > 1 else 0,
                       'Imbalance_Ratio': f'{imbalance_ratio(synth_counts):.2f}'})

# DRAMP: Activity as multi-label
dramp['has_antimicrobial'] = dramp['Activity'].str.contains('Antimicrobial', na=False, case=False)
antim_counts = dramp['has_antimicrobial'].value_counts()
imbalance_data.append({'Dataset': 'DRAMP', 'Label': 'has_antimicrobial',
                       'Majority': 'True' if antim_counts.idxmax() else 'False',
                       'Majority_Count': antim_counts.max(),
                       'Minority': 'False' if antim_counts.idxmin() else 'True',
                       'Minority_Count': antim_counts.min(),
                       'Imbalance_Ratio': f'{imbalance_ratio(antim_counts):.2f}'})

# DRAMP: Hemolysis availability
dramp['has_hemolysis'] = dramp['Hemolytic_activity'] != 'No hemolysis information or data found in the reference(s) presented in this entry'
hemo_counts = dramp['has_hemolysis'].value_counts()
imbalance_data.append({'Dataset': 'DRAMP', 'Label': 'has_hemolysis_data',
                       'Majority': 'No' if hemo_counts.idxmax() == False else 'Yes',
                       'Majority_Count': hemo_counts.max(),
                       'Minority': 'Yes' if hemo_counts.idxmin() == False else 'No',
                       'Minority_Count': hemo_counts.min(),
                       'Imbalance_Ratio': f'{imbalance_ratio(hemo_counts):.2f}'})

# DRAMP: Cytotoxicity availability
dramp['has_cytotoxicity'] = dramp['Cytotoxicity'] != 'No cytotoxicity information found'
cyto_counts = dramp['has_cytotoxicity'].value_counts()
imbalance_data.append({'Dataset': 'DRAMP', 'Label': 'has_cytotoxicity_data',
                       'Majority': 'No' if cyto_counts.idxmax() == False else 'Yes',
                       'Majority_Count': cyto_counts.max(),
                       'Minority': 'Yes' if cyto_counts.idxmin() == False else 'No',
                       'Minority_Count': cyto_counts.min(),
                       'Imbalance_Ratio': f'{imbalance_ratio(cyto_counts):.2f}'})

imbalance_df = pd.DataFrame(imbalance_data)
imbalance_df

In [ ]:
# Visualize imbalance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# DBAASP synthesis
synth_counts = dbaasp['synthesis_type'].value_counts()
axes[0, 0].bar(synth_counts.index, synth_counts.values, color=['#66b3ff', '#ff9999', '#99ff99', '#ffcc99'])
axes[0, 0].set_title(f'DBAASP - Synthesis Type (Ratio: {imbalance_ratio(synth_counts):.1f})')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# DRAMP has_antimicrobial
antim_counts = dramp['has_antimicrobial'].value_counts()
axes[0, 1].bar(['Not Antimicrobial', 'Antimicrobial'], antim_counts.values, color=['#ff9999', '#66b3ff'])
axes[0, 1].set_title(f'DRAMP - Antimicrobial Activity (Ratio: {imbalance_ratio(antim_counts):.1f})')
axes[0, 1].set_ylabel('Count')

# DRAMP hemolysis
hemo_counts = dramp['has_hemolysis'].value_counts()
axes[1, 0].bar(['No Data', 'Has Data'], hemo_counts.values, color=['#ff9999', '#66b3ff'])
axes[1, 0].set_title(f'DRAMP - Hemolysis Data Available (Ratio: {imbalance_ratio(hemo_counts):.1f})')
axes[1, 0].set_ylabel('Count')

# DRAMP cytotoxicity
cyto_counts = dramp['has_cytotoxicity'].value_counts()
axes[1, 1].bar(['No Data', 'Has Data'], cyto_counts.values, color=['#ff9999', '#66b3ff'])
axes[1, 1].set_title(f'DRAMP - Cytotoxicity Data Available (Ratio: {imbalance_ratio(cyto_counts):.1f})')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/raw/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Cross-Database Overlap Analysis

Check for sequence overlap between databases (important for train/test split design).

In [ ]:
# Get unique sequences from each database
dbaasp_seqs = set(dbaasp['sequence'].dropna().str.upper().unique())
apd3_seqs = set(apd3['sequence'].dropna().str.upper().unique())
dramp_seqs = set(dramp['Sequence'].dropna().str.upper().unique())

print(f'DBAASP unique sequences: {len(dbaasp_seqs)}')
print(f'APD3 unique sequences: {len(apd3_seqs)}')
print(f'DRAMP unique sequences: {len(dramp_seqs)}')

# Overlaps
dbaasp_apd3 = len(dbaasp_seqs & apd3_seqs)
dbaasp_dramp = len(dbaasp_seqs & dramp_seqs)
apd3_dramp = len(apd3_seqs & dramp_seqs)
all_three = len(dbaasp_seqs & apd3_seqs & dramp_seqs)

print(f'\nDBAASP ∩ APD3: {dbaasp_apd3}')
print(f'DBAASP ∩ DRAMP: {dbaasp_dramp}')
print(f'APD3 ∩ DRAMP: {apd3_dramp}')
print(f'All three: {all_three}')

In [ ]:
# Overlap visualization
from matplotlib_venn import venn3

plt.figure(figsize=(10, 8))
venn3([dbaasp_seqs, apd3_seqs, dramp_seqs],
       set_labels=('DBAASP', 'APD3', 'DRAMP'))
plt.title('Sequence Overlap Between Databases')
plt.savefig('../data/raw/database_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Summary & Key Findings

In [ ]:
print('=' * 80)
print('EDA SUMMARY')
print('=' * 80)
print(f'''
DATASET OVERVIEW:
  - DBAASP: {len(dbaasp):,} records, {len(dbaasp_seqs):,} unique sequences
  - APD3:   {len(apd3):,} records, {len(apd3_seqs):,} unique sequences
  - DRAMP:  {len(dramp):,} records, {len(dramp_seqs):,} unique sequences

SEQUENCE LENGTH:
  - DBAASP: mean={dbaasp['seq_length'].mean():.0f}, median={dbaasp['seq_length'].median():.0f}, range=[{dbaasp['seq_length'].min()}-{dbaasp['seq_length'].max()}]
  - APD3:   mean={apd3['sequence_length'].mean():.0f}, median={apd3['sequence_length'].median():.0f}, range=[{apd3['sequence_length'].min()}-{apd3['sequence_length'].max()}]
  - DRAMP:  mean={dramp['seq_length'].mean():.0f}, median={dramp['seq_length'].median():.0f}, range=[{dramp['seq_length'].min()}-{dramp['seq_length'].max()}]

CLASS IMBALANCE:
  - DBAASP synthesis type ratio: {imbalance_ratio(synth_counts):.1f}
  - DRAMP antimicrobial ratio: {imbalance_ratio(antim_counts):.1f}
  - DRAMP hemolysis data available: {hemo_counts[True] if True in hemo_counts.index else 0} / {len(dramp)} ({hemo_counts.get(True, 0)/len(dramp)*100:.1f}%)
  - DRAMP cytotoxicity data available: {cyto_counts[True] if True in cyto_counts.index else 0} / {len(dramp)} ({cyto_counts.get(True, 0)/len(dramp)*100:.1f}%)

OVERLAP:
  - DBAASP ∩ APD3:  {dbaasp_apd3} sequences
  - DBAASP ∩ DRAMP: {dbaasp_dramp} sequences
  - APD3 ∩ DRAMP:   {apd3_dramp} sequences
  - All three:      {all_three} sequences
""")

In [ ]:
print('All exploration complete. Key findings documented above.')